# Prompt Injection Experiment Analysis

Loads all JSON result files from `results/` and produces:
- Cross-model × attack success-rate heatmap
- Cross-domain × attack heatmap (Phase 3+)
- Attack effectiveness ranking
- Score distribution table


In [ ]:
import json
import glob
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120


## 1  Load Results


In [ ]:
def load_results(results_dir='results'):
    records = []
    files = sorted(glob.glob(f'{results_dir}/*.json'))
    for path in files:
        with open(path) as f:
            data = json.load(f)
        for record in data:
            record['source_file'] = Path(path).name
            records.append(record)
    print(f'Loaded {len(records)} results from {len(files)} file(s)')
    return records

records = load_results()


In [ ]:
df = pd.DataFrame(records)

# Drop error rows for analysis
df_clean = df[df['score'].notna() & ~df['score'].isin(['ERROR'])].copy()
df_clean['success_num'] = (df_clean['score'] == 'SUCCESS').astype(int)

print('Shape:', df_clean.shape)
print('Models: ', sorted(df_clean['model'].unique()))
print('Attacks:', sorted(df_clean['attack_id'].unique()))
print('Domains:', sorted(df_clean['domain'].unique()))
df_clean.head()


## 2  Cross-Model Comparison (Attack Success Rate)


In [ ]:
pivot_model = (
    df_clean
    .groupby(['model', 'attack_id'])['success_num']
    .mean()
    .mul(100)
    .round(1)
    .unstack('attack_id')
)
pivot_model


In [ ]:
fig, ax = plt.subplots(figsize=(13, max(3, len(pivot_model) * 0.9)))
sns.heatmap(
    pivot_model,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn_r',
    vmin=0,
    vmax=100,
    linewidths=0.4,
    cbar_kws={'label': 'Success Rate (%)'},
    ax=ax,
)
ax.set_title('Attack Success Rate (%) — Model x Attack', fontsize=14, pad=12)
ax.set_xlabel('Attack ID')
ax.set_ylabel('Model')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/heatmap_model_attack.png', dpi=150)
plt.show()


## 3  Cross-Domain Comparison

Populated after Phase 3 (finance + medical domains added).


In [ ]:
if df_clean['domain'].nunique() > 1:
    pivot_domain = (
        df_clean
        .groupby(['domain', 'attack_id'])['success_num']
        .mean()
        .mul(100)
        .round(1)
        .unstack('attack_id')
    )
    fig, ax = plt.subplots(figsize=(13, 3))
    sns.heatmap(
        pivot_domain,
        annot=True,
        fmt='.1f',
        cmap='RdYlGn_r',
        vmin=0,
        vmax=100,
        linewidths=0.4,
        cbar_kws={'label': 'Success Rate (%)'},
        ax=ax,
    )
    ax.set_title('Attack Success Rate (%) — Domain x Attack', fontsize=14, pad=12)
    ax.set_xlabel('Attack ID')
    ax.set_ylabel('Domain')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('../results/heatmap_domain_attack.png', dpi=150)
    plt.show()
else:
    print('Single domain loaded. Run finance + medical experiments to enable cross-domain view.')


## 4  Attack Effectiveness Ranking


In [ ]:
attack_stats = (
    df_clean
    .groupby('attack_id')['success_num']
    .agg(success_rate='mean', n='count')
    .assign(success_pct=lambda x: (x['success_rate'] * 100).round(1))
    .sort_values('success_pct', ascending=False)
)
attack_stats[['success_pct', 'n']]


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#d73027' if v >= 50 else '#4dac26' for v in attack_stats['success_pct']]
attack_stats['success_pct'].plot.bar(ax=ax, color=colors, edgecolor='white', width=0.6)
ax.axhline(50, linestyle='--', linewidth=1.2, color='gray', alpha=0.7, label='50% threshold')
ax.set_title('Overall Attack Success Rate (%)', fontsize=14)
ax.set_xlabel('Attack ID')
ax.set_ylabel('Success Rate (%)')
ax.set_ylim(0, 105)
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/bar_attack_effectiveness.png', dpi=150)
plt.show()


## 5  Score Distribution by Attack


In [ ]:
score_dist = (
    df_clean
    .groupby(['attack_id', 'score'])
    .size()
    .unstack(fill_value=0)
)
score_dist


## 6  Per-Model Attack Breakdown (stacked bars)


In [ ]:
model_attack = (
    df_clean
    .groupby(['model', 'attack_id'])['success_num']
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={'success_num': 'success_pct'})
)

models = sorted(model_attack['model'].unique())
n_models = len(models)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4), sharey=True)
if n_models == 1:
    axes = [axes]

for ax, model in zip(axes, models):
    subset = model_attack[model_attack['model'] == model].set_index('attack_id')
    subset['success_pct'].plot.bar(ax=ax, color='steelblue', edgecolor='white', width=0.6)
    ax.set_title(model.split('/')[-1], fontsize=11)
    ax.set_ylim(0, 105)
    ax.axhline(50, linestyle='--', linewidth=1, color='gray', alpha=0.6)
    ax.set_xlabel('')
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)

axes[0].set_ylabel('Success Rate (%)')
fig.suptitle('Attack Success Rate by Model', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/bar_per_model.png', dpi=150)
plt.show()


## 7  Export Baseline Table CSV


In [ ]:
pivot_model.to_csv('../results/baseline_table.csv')
print('Saved results/baseline_table.csv')
pivot_model
